# PlantVillage 实验图表（论文风格）

在 **Kaggle** 上运行：上传本 notebook + 将 **TensorBoard 的 `events.out.tfevents.*` 所在目录** 加到 Input（或放在 `/kaggle/working` 解压路径），修改下方路径后 **Run All**。

本 notebook 输出：**高分辨率 PNG**（默认 300 dpi）、可选 **CSV 导出**、与 **可选** 的验证集 F1/混淆矩阵（需提供权重与数据路径）。

In [ ]:
# --- 依赖 ---
!pip -q install matplotlib seaborn tensorboard pandas

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

# ---------- 论文风格全局设置 ----------
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update(
    {
        "font.family": "serif",
        "font.serif": ["DejaVu Serif", "Times New Roman", "Bitstream Vera Serif"],
        "font.size": 10,
        "axes.labelsize": 11,
        "axes.titlesize": 11,
        "legend.fontsize": 9,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
    }
)

OUT_DIR = Path("/kaggle/working/paper_figs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output:", OUT_DIR.resolve())

In [ ]:
#这里把文件复制过去
cp /kaggle/input/datasets/xing15/snn-data/events.out.tfevents.1778243770.9267771935b4.23.0 /kaggle/working/

In [ ]:
# ========== 修改为你的路径 ==========
# TensorBoard 日志目录（其下应有 events.out.tfevents.*）
TFEVENTS_DIR = Path("/kaggle/working/events.out.tfevents.1778240028.7acca14a756f.22.0")  # 或 "/kaggle/input/your-tb-logs/run1"

MAX_EPOCH = 15  # 只画前 N 个 epoch（step 0 .. N-1）
SMOOTH_VAL = 3  # val 滑动平均窗口（1=不平滑）
# val 准确率：raw_only | smooth_only（仅平滑）| both（原始淡色 + 平滑）
VAL_ACC_SHOW = "smooth_only"

# 可选：导出 CSV
EXPORT_CSV = True

In [ ]:
def load_tb_scalars(logdir: Path, max_step: int) -> dict:
    ea = EventAccumulator(str(logdir), size_guidance={"scalars": 0})
    ea.Reload()
    tags = ea.Tags().get("scalars", [])
    out = {}
    for tag in ("loss/train", "acc/train", "acc/val"):
        if tag not in tags:
            continue
        out[tag] = [(int(s.step), float(s.value)) for s in ea.Scalars(tag) if int(s.step) < max_step]
    return out


def rolling_mean(y, k: int):
    if k <= 1:
        return np.array(y, dtype=float)
    y = np.array(y, dtype=float)
    pad = k // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(k) / k
    return np.convolve(yp, ker, mode="valid")[: len(y)]


series = load_tb_scalars(TFEVENTS_DIR, MAX_EPOCH)
assert series, f"No scalars under {TFEVENTS_DIR}; tags found: try parent folder of events file"
print("Loaded:", {k: len(v) for k, v in series.items()})

In [ ]:
# ---------- 图 1：Loss + Accuracy（双栏论文常用） ----------
fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.8))

lt = np.array(series["loss/train"]) if "loss/train" in series else None
if lt is not None:
    axes[0].plot(lt[:, 0], lt[:, 1], color="#1f77b4", lw=1.8, label="Train loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Cross-entropy loss")
    axes[0].set_title("(a) Training loss")
    axes[0].legend(frameon=True)

at = np.array(series["acc/train"]) if "acc/train" in series else None
av = np.array(series["acc/val"]) if "acc/val" in series else None
if at is not None and av is not None:
    axes[1].plot(at[:, 0], 100 * at[:, 1], color="#2ca02c", lw=1.8, label="Train acc.")
    av_pct = 100 * av[:, 1]
    if VAL_ACC_SHOW in ("raw_only", "both"):
        axes[1].plot(
            av[:, 0],
            av_pct,
            color="#d62728",
            lw=1.4,
            alpha=0.38 if VAL_ACC_SHOW == "both" else 0.9,
            label="Val (raw)",
        )
    if VAL_ACC_SHOW in ("smooth_only", "both") and SMOOTH_VAL > 1:
        sm = rolling_mean(av[:, 1], SMOOTH_VAL)
        axes[1].plot(
            av[:, 0],
            100 * sm,
            color="#ff7f0e",
            ls="-" if VAL_ACC_SHOW == "smooth_only" else "--",
            lw=1.75,
            label=f"Val (MA-{SMOOTH_VAL})",
        )
    elif VAL_ACC_SHOW == "smooth_only" and SMOOTH_VAL <= 1:
        axes[1].plot(av[:, 0], av_pct, color="#d62728", lw=1.6, label="Val acc")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy (%)")
    axes[1].set_ylim(0, 102)
    axes[1].set_title("(b) Classification accuracy")
    axes[1].legend(loc="lower right", frameon=True)

fig.tight_layout()
p1 = OUT_DIR / "fig1_loss_acc.pdf"
p1_png = OUT_DIR / "fig1_loss_acc.png"
fig.savefig(p1_png)
try:
    fig.savefig(p1)
except Exception as e:
    print("PDF skip:", e)
plt.show()
print("Saved", p1_png)

In [ ]:
# ---------- 图 2：Train–Val 差距（过拟合可视化） ----------
if "acc/train" in series and "acc/val" in series:
    at = np.array(series["acc/train"])
    av = np.array(series["acc/val"])
    gap = 100 * (at[:, 1] - av[:, 1])
    fig, ax = plt.subplots(figsize=(3.6, 2.6))
    ax.bar(at[:, 0], gap, color="#9467bd", alpha=0.85, width=0.65)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Train acc. − Val acc. (pts)")
    ax.set_title("Generalization gap")
    fig.tight_layout()
    p2 = OUT_DIR / "fig2_train_val_gap.png"
    fig.savefig(p2)
    plt.show()
    print("Saved", p2)

In [ ]:
# ---------- 表格 + CSV ----------
rows = []
for i in range(MAX_EPOCH):
    row = {"epoch": i}
    for tag in ("loss/train", "acc/train", "acc/val"):
        if tag not in series:
            continue
        mp = {int(s): v for s, v in series[tag]}
        row[tag.replace("/", "_")] = mp.get(i, np.nan)
    rows.append(row)

df = pd.DataFrame(rows)
if "acc_train" in df.columns:
    df["acc_train_pct"] = 100 * df["acc_train"]
if "acc_val" in df.columns:
    df["acc_val_pct"] = 100 * df["acc_val"]
display(df.round(4))

if EXPORT_CSV:
    csvp = OUT_DIR / "metrics_first_epochs.csv"
    df.to_csv(csvp, index=False)
    print("Saved", csvp)

summary = {
    "best_val_acc": float(df["acc_val"].max()) if "acc_val" in df else None,
    "last_val_acc": float(df["acc_val"].iloc[-1]) if "acc_val" in df else None,
    "mean_val_last5": float(df["acc_val"].tail(5).mean()) if "acc_val" in df and len(df) >= 5 else None,
}
with open(OUT_DIR / "summary_metrics.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

## 可选：混淆矩阵 / F1（需要 PyTorch + 数据 + 权重）

若已 **Add Data** PlantVillage 且 **checkpoint** 在 `/kaggle/working`：下一格设 **`RUN_CONFUSION = True`**，并设 **`CONFUSION_KIND`**：

- **`"ann"`**：`build_ann_model`（Res2Net-ANN）或 checkpoint 带 **`backbone=torchvision_resnet18`** 时用 **TV ResNet-18 ANN**
- **`"snn_res2net"`**：`build_model` + `encode_batch`（`[T,B,C,H,W]`）
- **`"snn_resnet18"`**：`build_spiking_resnet18` + `encode_batch`（与训练一致）

否则保持 **`RUN_CONFUSION = False`** 跳过。

In [ ]:
# --- 拉取 GitHub 代码（把 URL 换成你的仓库） ---
import os

os.chdir('/kaggle/working')

GITHUB_URL = "https://github.com/xing11234/plantvillage_snn.git"  # 修改
BRANCH = "master"  # 或 master
CLONE_DIR = "/kaggle/working/plantvillage_snn"
!rm -rf {CLONE_DIR}
!git clone --depth 1 --branch {BRANCH} {GITHUB_URL} {CLONE_DIR}

os.chdir(CLONE_DIR)

print("Working dir:", os.getcwd())

In [ ]:
RUN_CONFUSION = False  # True：画混淆矩阵（需填 CKPT / DATA_ROOT）
# "ann" | "snn_res2net" | "snn_resnet18" — 须与 checkpoint 一致；ResNet-18 SNN 用 snn_resnet18
CONFUSION_KIND = "snn_resnet18"

CKPT = Path("/kaggle/working/best_msf_res2net.pt")
DATA_ROOT = Path("/kaggle/input/datasets/abdallahalidev/plantvillage-dataset")

if RUN_CONFUSION:
    import os
    import sys

    CODE_DIR = Path("/kaggle/working/plantvillage_snn")
    if CODE_DIR.is_dir():
        sys.path.insert(0, str(CODE_DIR))
        os.chdir(CODE_DIR)

    import torch
    from dataclasses import fields, replace
    from sklearn.metrics import classification_report, confusion_matrix

    from config import TrainConfig
    from data.kaggle_dataloader import get_kaggle_dataloaders

    ckpt = torch.load(CKPT, map_location="cpu", weights_only=False)
    d = ckpt["cfg"]
    names = {f.name for f in fields(TrainConfig)}
    cfg = replace(TrainConfig(), **{k: v for k, v in d.items() if k in names})

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    _, val_loader, nc = get_kaggle_dataloaders(
        str(DATA_ROOT),
        image_size=cfg.image_size,
        batch_size=64,
        num_workers=2,
        val_ratio=0.2,
        seed=cfg.seed,
        auto_find_subdir=True,
    )
    cfg.num_classes = nc

    if CONFUSION_KIND == "ann":
        bb = str(ckpt.get("backbone", "")).lower()
        if "torchvision_resnet18" in bb or "resnet18" in CKPT.name.lower():
            from models.resnet18_tv_ann import build_tv_resnet18_ann_from_config

            model = build_tv_resnet18_ann_from_config(cfg, imagenet_pretrained=False).to(device)
        else:
            from models.res2net_ann import build_ann_model

            model = build_ann_model(cfg).to(device)
        model.load_state_dict(ckpt["model"], strict=True)
        model.eval()
        ys, ps = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(device, non_blocking=True)
                logits = model(x)
                ps.extend(logits.argmax(1).cpu().tolist())
                ys.extend(y.tolist())
    elif CONFUSION_KIND in ("snn_res2net", "snn_resnet18"):
        from train import encode_batch
        from utils.train_utils import reset_snn_state

        if CONFUSION_KIND == "snn_resnet18":
            from models.spiking_resnet18_backbone import build_spiking_resnet18

            model = build_spiking_resnet18(cfg).to(device)
        else:
            from models.res2net_msf import build_model

            model = build_model(cfg).to(device)
        model.load_state_dict(ckpt["model"], strict=True)
        model.eval()
        ys, ps = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                x = encode_batch(images, cfg.T, device)
                reset_snn_state(model)
                logits_t = model(x)
                logits = logits_t.mean(dim=0)
                pred = logits.argmax(dim=1)
                ps.extend(pred.cpu().tolist())
                ys.extend(labels.cpu().tolist())
    else:
        raise ValueError(f"Unknown CONFUSION_KIND={CONFUSION_KIND!r}")

    cm = confusion_matrix(ys, ps, labels=list(range(nc)))
    fig, ax = plt.subplots(figsize=(8, 6.5))
    sns.heatmap(cm, cmap="Blues", square=False, ax=ax, cbar_kws={"label": "Count"})
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"Confusion matrix (val) — {CONFUSION_KIND}")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "fig3_confusion_matrix.png", dpi=300)
    plt.show()

    rep = classification_report(ys, ps, labels=list(range(nc)), output_dict=True, zero_division=0)
    with open(OUT_DIR / "classification_report.json", "w", encoding="utf-8") as f:
        json.dump(rep, f, indent=2)
    print("macro F1", rep["macro avg"]["f1-score"])
else:
    print("Skip confusion: set RUN_CONFUSION=True and CONFUSION_KIND / CKPT / DATA_ROOT.")

## 下载

在 Kaggle **Output** 中打包 `/kaggle/working/paper_figs/` 下载：`fig1_loss_acc.png`（及可选 pdf）、`fig2_train_val_gap.png`、`metrics_first_epochs.csv`、`summary_metrics.json`。